<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/3D_conf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 48.8 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import os

def process_and_embed_mols(dataframe, smiles_col, id_col, output_writer, num_confs=5):
    """
    Scientifically accurate function to generate 3D ensembles.
    Isolates conformers into unique single-conformer molecule objects
    to guarantee flawless property assignment and zero index clashing.
    """
    unique_molecules_saved = 0

    for index, row in dataframe.iterrows():
        smiles = row[smiles_col]

        # Extract ID
        if id_col in dataframe.columns and pd.notna(row[id_col]):
            mol_id = str(row[id_col])
        else:
            mol_id = f"Mol_{index}"

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: Cannot parse SMILES for {mol_id}. Skipping.")
            continue

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # Enforce ETKDGv3 and chirality
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        params.enforceChirality = True

        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=num_confs, params=params)
        if not conf_ids:
            print(f"Warning: 3D embedding failed for {mol_id}. Skipping.")
            continue

        molecule_had_successful_conformer = False

        for conf_id in conf_ids:
            # Maximize iterations (500) to prevent throwing away bulky Enamine chemical space
            opt_status = AllChem.MMFFOptimizeMolecule(mol, confId=conf_id, maxIters=500)

            if opt_status == 1:
              print("Did not conform")
            if opt_status == 0:
                # Create a clean, blank copy of the topology (no conformers attached)
                single_conf_mol = Chem.Mol(mol)
                single_conf_mol.RemoveAllConformers()

                # Extract the specific optimized conformer from the parent molecule
                conf = mol.GetConformer(conf_id)

                # Force internal ID to 0 so SDWriter reads it cleanly
                conf.SetId(0)

                # Add the re-indexed conformer back into the isolated wrapper
                single_conf_mol.AddConformer(conf, assignId=False)

                # Set distinct, isolated metadata properties that won't leak or overwrite
                single_conf_mol.SetProp("_Name", f"{mol_id}_conf_{conf_id}")
                single_conf_mol.SetProp("Parent_ID", mol_id)
                single_conf_mol.SetProp("Activity_Class", str(row.get('Activity_Class', 'Unknown')))

                # Safely write (it sees exactly one conformer sitting at index 0)
                output_writer.write(single_conf_mol)
                molecule_had_successful_conformer = True

        if molecule_had_successful_conformer:
            unique_molecules_saved += 1

    return unique_molecules_saved


# Data Loading

print("Loading datasets...")
sheet1_path = "/content/AA_Derivatives_Actives.csv"
sheet2_path = "/content/True_Negatives.csv"
sheet3_path = "/content/Enamine_VS_Hits_TrueNegs.csv"

df_sheet1_actives = pd.read_csv(sheet1_path)
df_neg = pd.read_csv(sheet2_path)
df_vs = pd.read_csv(sheet3_path)


# Chemoinformatic Standardisation

print("Generating canonical SMILES representations to prevent data leakage...")
def safe_canonicalize(smiles_str):
    if pd.isna(smiles_str):
        return None
    mol = Chem.MolFromSmiles(str(smiles_str).strip())
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

df_sheet1_actives['Canonical_SMILES'] = df_sheet1_actives['SMILES'].apply(safe_canonicalize)
df_neg['Canonical_SMILES'] = df_neg['SMILES'].apply(safe_canonicalize)
df_vs['Canonical_SMILES'] = df_vs['SMILES'].apply(safe_canonicalize)


# Data Curation and Strict Subset Filtering

print("Enforcing strict activity definitions and handling missing data...")

# Parse and clean Sheet 1, tossing out 'No data' text rows
extracted_ki = df_sheet1_actives['Exp Ki (µM)'].astype(str).str.split('±').str[0]
df_sheet1_actives['Numeric_Ki'] = pd.to_numeric(extracted_ki, errors='coerce')
df_sheet1_actives = df_sheet1_actives.dropna(subset=['Numeric_Ki', 'Canonical_SMILES']).copy()
df_sheet1_actives['Activity_Class'] = 1

# Explicit white-list filtering for Sheet 3 actives to completely bypass NaN rows
valid_active_tiers = ['≤ 1 µM', '≤ 10 µM', '≤ 100 µM']
df_sheet3_actives = df_vs[df_vs['Activity Tier'].isin(valid_active_tiers)].copy()
df_sheet3_actives = df_sheet3_actives.dropna(subset=['Canonical_SMILES'])
df_sheet3_actives['Activity_Class'] = 1

# Isolate true negatives
df_vs_inactives = df_vs[df_vs['Activity Tier'] == 'No Inhibition'].copy()
df_vs_inactives = df_vs_inactives.dropna(subset=['Canonical_SMILES'])
df_vs_inactives['Activity_Class'] = 0

df_neg = df_neg.dropna(subset=['Canonical_SMILES']).copy()
df_neg['Activity_Class'] = 0


# Zero Leakage Cross Reference Verification

# Establish cross-referencing pools using strictly canonicalized chemical space
sheet1_smiles = set(df_sheet1_actives['Canonical_SMILES'])
sheet3_active_smiles = set(df_sheet3_actives['Canonical_SMILES'])
all_active_smiles = sheet1_smiles | sheet3_active_smiles

# Execute perfect deduplication
df_sheet3_actives = df_sheet3_actives[~df_sheet3_actives['Canonical_SMILES'].isin(sheet1_smiles)]
df_neg = df_neg[~df_neg['Canonical_SMILES'].isin(all_active_smiles)]
df_vs_inactives = df_vs_inactives[~df_vs_inactives['Canonical_SMILES'].isin(all_active_smiles)]

print("Data deduplication complete. Proceeding to 3D embedding...")


# 3D Embedding and Export

# Set 1: Training Actives
writer_train_actives = Chem.SDWriter("Training_Actives_3D.sdf")
act_count_1 = process_and_embed_mols(df_sheet1_actives, 'Canonical_SMILES', 'Compound ID', writer_train_actives, num_confs=5)
act_count_2 = process_and_embed_mols(df_sheet3_actives, 'Canonical_SMILES', 'Catalog ID', writer_train_actives, num_confs=5)
writer_train_actives.close()

total_actives = act_count_1 + act_count_2
print(f"Set 1 Complete: {total_actives} unique molecules saved for Training (ACTIVES).")

# Set 2: Training Negatives
writer_train_negatives = Chem.SDWriter("Training_Negatives_3D.sdf")
neg_count_s2 = process_and_embed_mols(df_neg, 'Canonical_SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
neg_count_s3 = process_and_embed_mols(df_vs_inactives, 'Canonical_SMILES', 'Catalog ID', writer_train_negatives, num_confs=5)
writer_train_negatives.close()

total_negatives = neg_count_s2 + neg_count_s3
print(f"Set 2 Complete: {total_negatives} unique molecules saved for Training (NEGATIVES).")

Loading datasets...
Generating canonical SMILES representations to prevent data leakage...
Enforcing strict activity definitions and handling missing data...
Data deduplication complete. Proceeding to 3D embedding...
Set 1 Complete: 83 unique molecules saved for Training (ACTIVES).
Set 2 Complete: 54 unique molecules saved for Training (NEGATIVES).
